### Setup environment

In [5]:
%pip install accelerate bitsandbytes torch matplotlib unsloth peft trl scikit-learn tokenizers transformers evaluate python-dotenv

Note: you may need to restart the kernel to use updated packages.


### import

In [6]:
import os
import sys
import torch
import evaluate
import numpy as np
import torch.nn as nn
from trl import SFTTrainer
from tqdm.auto import tqdm
from dotenv import load_dotenv
from datasets import load_dataset
from huggingface_hub import login
from accelerate import Accelerator
from unsloth import FastLanguageModel
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoConfig, TrainingArguments

### Login huggingface

In [7]:
load_dotenv()
hf_token = os.getenv("HUGFACE_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Successfully logged in to Hugging Face.")
else:
    print("Hugging Face token not found. Please set the HUGFACE_TOKEN environment variable.")

Successfully logged in to Hugging Face.


### Device

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Load dataset

In [9]:
print("Loading dataset...")
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
print("Dataset loaded successfully.")
dataset

Loading dataset...
Dataset loaded successfully.


Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 15011
})

### Processing

In [11]:
def format_conversation(examples):
    text = []
    instruction = examples["instruction"]
    context = examples["context"]
    response = examples["response"]
    for instruction, context, response in zip(instruction, context, response):
        conversation = f"<|im_start|>user\n{instruction}\n{context}<|im_end|>\n<|im_start|>assistant\n{response}<|im_end|>"
        text.append(conversation)
    return {"text": text}
dataset = dataset.map(format_conversation, batched=True)

Map: 100%|██████████| 15011/15011 [00:00<00:00, 67835.71 examples/s]


In [12]:
dataset[0]

{'instruction': 'When did Virgin Australia start operating?',
 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.",
 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.',
 'category': 'closed_qa',
 'text': "<|im_start|>user\nWhen did Virgin Australia start operating?\nVirgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services

### Load model

In [13]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    max_seq_length=2048,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    init_lora_weights="gaussian",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model

==((====))==  Unsloth 2025.11.1: Fast Qwen2 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.999 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.11.1 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
            

### Setup train and training

In [20]:

dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_data = dataset_split["train"]
eval_data = dataset_split["test"]

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data, 
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=2048,
    num_train_epochs=3,
    packing=True,
    args=TrainingArguments(
        output_dir="./qwen-2.5b-finetuned-qlora",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        save_total_limit=1,
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        learning_rate=2e-4,
        weight_decay=0.01,
        push_to_hub=True,
    )
)


Unsloth: Tokenizing ["text"] (num_proc=32): 100%|██████████| 1502/1502 [00:07<00:00, 189.71 examples/s]


In [21]:
trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 13,509 | Num Epochs = 3 | Total steps = 2,535
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,2.178200,2.121029
2,1.925100,2.116009
3,1.757100,2.150738


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


TrainOutput(global_step=2535, training_loss=1.9534719493497288, metrics={'train_runtime': 8279.473, 'train_samples_per_second': 4.895, 'train_steps_per_second': 0.306, 'total_flos': 3.341211971073408e+16, 'train_loss': 1.9534719493497288, 'epoch': 3.0})

### Test

In [22]:
import torch
from unsloth import FastLanguageModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="qwen-2.5b-finetuned-qlora",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
)
FastLanguageModel.for_inference(model)


==((====))==  Unsloth 2025.11.1: Fast Qwen2 patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.999 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896, padding_idx=151654)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
    

In [23]:
def generate_response(prompt):
    formatted_prompt = f"<|im_start|>user\n{prompt}\n<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_length=2048, 
            do_sample=False, 
            repetition_penalty=1.15, 
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    return response


test_prompt = "What is the capital of France?"
response = generate_response(test_prompt)
print("Prompt:", test_prompt)
print("Response:", response)

Prompt: What is the capital of France?
Response: The capital of France is Paris.
